In [2]:
from clickhouse_connect import get_client
from datetime import date
import time
# Библиотеки для отправки почты
import smtplib # подключение к почте через протокол SMTP
from email.mime.text import MIMEText

import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
smtp_token = os.getenv("SMTP_TOKEN")

In [4]:
target_table = 'customer.sales'
today = date.today()
recipient_email = 'kholkinnik@mail.ru' # указать ящик куда хоти отправить 
smtp_host = 'smtp.mail.ru'
smtp_port = 587
smtp_user = 'kholkinnik@mail.ru' # указть ящик с которого хотим отправить 
#smtp_token = 'UCFeGffAV1DAogRnY2vO'


client = get_client(host='localhost', port=8123, username='user', password='strongpassword')
data_found = False

for attempt in range(1, 5):
    print(f"\n🔄 Попытка {attempt} из 4")

    result = client.query(f"""
        SELECT count() FROM customer.imports
        WHERE table_name = '{target_table}'
          AND last_import_date = toDate('{today}')
    """).result_rows[0][0]

    if result > 0:
        print(f"✅ Данные за {today} в {target_table} присутствуют.")
        data_found = True
        break
    else:
        print(f"❌ Данных за {today} в {target_table} нет.")
        if attempt < 4:
            print("⏳ Ждём 1 час до следующей попытки...")
            time.sleep(1)  # ждать 1 час

if not data_found:
    subject = f"[ALERT] Нет данных в {target_table} за {today}"
    body = f"Данных в таблице {target_table} за {today} так и не появилось после 4 попыток проверки."

    msg = MIMEText(body)
    msg["Subject"] = subject
    msg["From"] = smtp_user
    msg["To"] = recipient_email

    try:
        with smtplib.SMTP(smtp_host, smtp_port) as server:
            server.starttls()
            server.login(smtp_user, smtp_token)
            server.send_message(msg)
        print(f"📧 Уведомление отправлено на {recipient_email}")
    except Exception as e:
        print(f"❌ Ошибка при отправке письма: {e}")


🔄 Попытка 1 из 4
❌ Данных за 2025-11-24 в customer.sales нет.
⏳ Ждём 1 час до следующей попытки...

🔄 Попытка 2 из 4
❌ Данных за 2025-11-24 в customer.sales нет.
⏳ Ждём 1 час до следующей попытки...

🔄 Попытка 3 из 4
❌ Данных за 2025-11-24 в customer.sales нет.
⏳ Ждём 1 час до следующей попытки...

🔄 Попытка 4 из 4
❌ Данных за 2025-11-24 в customer.sales нет.
📧 Уведомление отправлено на kholkinnik@mail.ru
